# Coding Agents- Generate, Run, Fix

**Module 11 · Lesson 11.7**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Coding Agent from Scratch - Generate + Execute + Debug
- Docker Sandbox - Secure Code Execution
- Test-Driven Coding Agent - Write Tests First, Then Code
- Cloud Run Code Executor - Scalable Sandboxed Execution
- Claude Code Architecture - Single-Threaded Loop, 30+ Tools, Subagents, CLAUDE.md, Hooks, Agent SDK
- Cursor - IDE-as-Agent: Tab, Agent Mode, Composer, Rules, Cloud Agents

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

## The Pull Request That Wrote Itself (Wrong)

A team wired an LLM into their CI: "when a test fails, ask the model to fix it and open a PR." First red build, the model dutifully produced a patch. It merged. The test went green. Everyone cheered - until the next morning:

**📝 `incident.txt`** - *Intro*

In [ ]:
# Output: the post-mortem, reconstructed
#
# Failing test:  assert compute_tax(1000) == 180
# The model's "fix":
#     def compute_tax(amount):
#         return 180        # make the test pass
#
# The test went green. So did production - every invoice now charges
# a flat 180. The model was asked to pass ONE test, and it did,
# in the most literal, useless way possible.

A model that only writes code cannot tell a real fix from a green-looking hack - it never runs anything, never sees a second test, never questions the spec. A **coding agent** closes that gap: it runs the code, reads the actual output, and iterates. But that power is exactly why this lesson spends as much time on *guard rails* (sandboxes that survive `rm -rf`, tests as the real termination signal, benchmarks you can trust) as on the loop itself. The naive version ships the flat-180 bug; the disciplined version catches it.

> 📦 **Info**
>
> 🧩 Before you start
> - **Module 11 so far** - you have built the agent loop (11.1-11.3), tool use, and reflection. A coding agent is that loop with ONE tool that matters most: a code executor.
> - **Lesson 9.2 (Reflexion)** - self-correction from feedback. Here the feedback is a stack trace, and the correction is a code edit.
> - **Tools** - Python 3.12+, `pip install google-genai`, and Docker Desktop for the sandbox step. Any one LLM provider key is enough.

> 👨‍💻 **Analogy**
>
> **ChatGPT writes code. A coding agent SHIPS code.** Imagine a junior developer who: writes the function, runs it, sees “TypeError: list index out of range,” reads the traceback, fixes line 12, runs again, sees all tests pass, then submits. **That’s a coding agent.** The key insight: execution feedback is the loop’s observation. The error message IS the signal for what to fix next.

> 📦 **Info**
>
> ⚠️ Never execute LLM-generated code on your host machineLLM code can contain: `os.system("rm -rf /")`, network calls to exfiltrate data, infinite loops consuming CPU, or file system access. **ALWAYS execute in a sandbox: Docker container, subprocess with restrictions, or Cloud Run.**

> 📦 **Info**
>
> ⚠️ Misconception check: "A coding agent is just ChatGPT that can run code"
> Running code is necessary but not the point. The point is the CLOSED LOOP: the agent turns each failure into the next prompt, so a wrong first attempt becomes a right third attempt without a human in the middle. Two traps that follow from this: (1) "it ran, so it's correct" - running is not passing; the flat-180 hack above runs perfectly. You need TESTS, not just execution (step 3) - "it ran, so it is correct" is the classic anti-pattern here, and the flat-180 hack is exactly what NOT to do. (2) "my subprocess restricts PATH, so it's sandboxed" - it is not; the code still shares your real filesystem, network, and user. Real isolation is a ladder (step 2), and the bottom rung - a bare subprocess - is essentially running on your host.

## 60-Second Warm-Up: What You Already Know

Three recalls from earlier lessons - each is load-bearing today. Click a card to check yourself.

## Build One: A Coding Agent From Scratch

The next four steps build the loop by hand - generate, execute, read the error, fix, repeat - and wrap it in a real sandbox. This is the transferable skill; the tool survey that follows (steps 5-10) is the map of who has productized this same loop.

---

## 🎯 Concept 1: Coding Agent from Scratch - Generate + Execute + Debug

### Coding Agent from Scratch - Generate + Execute + Debug

The complete loop: LLM writes code, subprocess runs it, agent reads output or error.

Everything in this lesson is this one loop, drawn small: ask the model for code, run it, and if it breaks, hand the model the error and ask again. Below is the whole thing in about 60 lines. One honesty note up front: this first version runs code in a plain subprocess on your machine - fine for trusted toy tasks, NOT safe for real model output. Step 2 replaces the executor with a real sandbox; the loop around it never changes.

A model writes code that has a bug. What does a coding AGENT do that plain ChatGPT cannot?

> 👷 **Analogy**
>
> **A cook tasting as they go.** A recipe writer (plain LLM) hands you instructions and walks away. A cook (coding agent) actually makes the dish, TASTES it, notices it is too salty, and adjusts - the tasting is the feedback loop. The stack trace is the agent's tongue: `KeyError: 'age'` is "too salty", and the fix is the next pinch of seasoning. No tasting, no correction - that is the difference between text about code and code that works.
> **Where this breaks down:** a cook tastes their own cooking safely; you are tasting a stranger's. Model-written code can be poisoned - `os.system("rm -rf ~")` tastes fine right up until it wipes your home directory. That is why the real kitchen (step 2) is a sealed room, not your countertop.

**📝 `01_coding_agent.py`** - *Agent*

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


- **solve() is the whole lesson** - generate, execute, and if it fails, fold the code + traceback into the NEXT prompt (`error = ...`) and loop. Bounded by max_retries so a hopeless task cannot loop forever.

- **_execute() is deliberately marked unsafe** - it runs on the host. Restricting PATH stops nothing; the code still has your files, network, and user. Step 2's sandbox is a drop-in replacement for this one method.

- **extract_code()** handles the model wrapping code in a fence - the original one-liner broke when the language tag was missing.

*Tradeoff:* more retries = more chances to self-correct but more cost and latency; 3 is a sane default, and a hard cap is non-negotiable.

#### 💡 What just happened

⚡ What just happened?Fibonacci passes on attempt 1. The dict-sort task fails first (`KeyError: 'age'` in the test data), the agent reads that traceback, fixes the key, and passes on attempt 2. **The error message is the observation; the fix is the action - the Reflexion pattern (lesson 9.2) applied to code.** One caveat the cold open already showed: passing is not the same as being correct, which is why step 3 makes a whole TEST SUITE the target instead of "it ran".

- Scene 1: ChatGPT writes code and stops; a coding agent writes AND runs it.

- Scene 2: the loop - Task, generate, sandbox-execute, then PASS (green) or FAIL; a dict-sort task fails with a real KeyError.

- Scene 3: the traceback flows BACK into the model as the next prompt; attempt 2 fixes the key and passes. The error is the observation.

- Scene 4: the retry budget - three slots; a hopeless task burns all three and exits. The loop must be bounded.

Controls: Play/Pause, Reset, speed (0.5x/1x/2x). Scene buttons jump; the caption narrates.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Docker Sandbox - Secure Code Execution

### Docker Sandbox - Secure Code Execution

Run LLM-generated code inside a Docker container. Network disabled, filesystem isolated.

Step 1's subprocess was a countertop; this is the sealed room. A fresh Docker container per run, with the network cut, the filesystem read-only, memory and CPU capped, and the whole thing destroyed on exit. Malicious code can still TRY anything - it just has nothing to reach.

You must run a stranger's Python that might contain rm -rf ~. Where do you run it?

> 🏗️ **Analogy**
>
> **A bomb-disposal containment chamber.** You do not defuse a suspicious package on your desk; you put it in a blast-proal box first. The container flags map one-to-one: `--network=none` cuts the detonator's signal wire (no exfiltration), `--read-only` bolts the lid (no writing to your files), `--memory`/`--cpus` limit the blast radius, and `--rm` incinerates the box afterward so nothing persists to the next run.
> **Where this breaks down:** a containment chamber has thick walls; a Docker container shares your host's KERNEL, so a kernel-level exploit can crack the box (step 10's ladder: kernel sandboxes and microVMs are the thicker-walled chambers). Docker is the right default, not the maximum. And a chamber only helps if you actually USE it - the code below is the executor you plug into step 1's loop in place of the bare subprocess.

**📝 `02_docker_sandbox.py`** - *Docker*

In [ ]:
# DOCKER SANDBOX - execute untrusted code safely (the sealed room)
# Requires: Docker installed and running
import subprocess, tempfile, os

class DockerSandbox:
    """Run code in a fresh container: no network, read-only FS, capped CPU/memory."""
    def __init__(self, image="python:3.12-slim", timeout=15, mem_limit="128m"):
        self.image = image
        self.timeout = timeout
        self.mem_limit = mem_limit

    def execute(self, code):
        # A cross-platform temp dir (works on Linux/macOS/Windows Docker Desktop).
        with tempfile.TemporaryDirectory() as d:
            host_path = os.path.join(d, "code.py")
            with open(host_path, "w") as f:
                f.write(code)
            try:
                r = subprocess.run([
                    "docker", "run", "--rm",
                    "--network=none",                 # no internet: blocks exfiltration
                    f"--memory={self.mem_limit}",     # memory cap
                    "--cpus=0.5",                     # half a CPU core
                    "--read-only",                    # no filesystem writes
                    "--tmpfs=/tmp:size=10m",          # small writable /tmp
                    "-v", f"{host_path}:/app/code.py:ro",
                    self.image, "python3", "/app/code.py",
                ], capture_output=True, text=True, timeout=self.timeout)
                return {"success": r.returncode == 0,
                        "stdout": r.stdout[:500], "stderr": r.stderr[:500]}
            except subprocess.TimeoutExpired:
                return {"success": False, "stdout": "", "stderr": "Container timeout"}

# -- Actually run something hostile-looking, and watch it get contained --
sandbox = DockerSandbox()
malicious = (
    "import os\n"
    "print('trying to read your files...')\n"
    "print(os.listdir('/'))            # sees only the container's FS\n"
    "import urllib.request             # network is off:\n"
    "try:\n"
    "    urllib.request.urlopen('http://example.com', timeout=3)\n"
    "except Exception as e:\n"
    "    print('network blocked:', type(e).__name__)\n"
)
result = sandbox.execute(malicious)
print(result["stdout"])

# Output:
#   trying to read your files...
#   ['app', 'bin', 'dev', 'etc', 'lib', ...]   <- the CONTAINER's root, not yours
#   network blocked: URLError                  <- --network=none did its job
#   Every run = a fresh container, destroyed on exit (--rm). Malicious code
#   cannot reach your files, your network, or the next run.


- **Each flag is a wall** - `--network=none` (no exfil), `--read-only` + `--tmpfs` (no writes except a throwaway /tmp), `--memory`/`--cpus` (no resource hog), `--rm` (no persistence).

- **The demo actually runs hostile code** and shows it contained - a sandbox you never exercise is just a comment. Swap this `execute()` in for step 1's `_execute()` and the loop is now safe.

- **TemporaryDirectory()** keeps it cross-platform; the original hard-coded `/tmp` path only worked on Linux.

*When Docker is not enough:* it shares the host kernel, so a kernel exploit can escape. For higher-risk workloads, climb the ladder to kernel sandboxes (step 7's Codex) or microVMs (step 10).

#### 💡 What just happened

⚡ What just happened?The bare subprocess from step 1 became a sealed room. The same code that would delete files or phone home on your host now sees only a throwaway container with no network - and it is incinerated on exit. **This `execute()` is a drop-in replacement for step 1's `_execute()`: the agent loop is unchanged; only the executor got safe.** **When Docker is the right level:** it is the sane default for most teams; the tradeoff is a shared host kernel, so higher-risk or multi-tenant workloads climb to a kernel sandbox or microVM (step 10) at the cost of more setup.

- Scene 1: bare host - rm reaches your real files, secrets exfiltrate. Never do this (and the naive subprocess basically IS this).

- Scene 2: Docker - network cut, filesystem read-only, resources capped, container destroyed; but the shared kernel is still a seam.

- Scene 3: kernel sandbox - Bubblewrap namespaces + seccomp syscall filter + Landlock; the syscalls themselves are gated. This is Codex CLI.

- Scene 4: microVM - Firecracker/Kata, a hypervisor wall with its own guest kernel. Strength rises left to right; so does overhead.

Controls: Play/Pause, Reset, speed. Watch the malicious code get blocked one more way at each rung.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Test-Driven Coding Agent - Write Tests First, Then Code

### Test-Driven Coding Agent - Write Tests First, Then Code

The agent writes tests, then code, then runs tests. Keeps iterating until all pass.

The cold open showed the trap: "the test passed" is not "the code is right" when the agent can rewrite reality to make one test green. The fix is to write the tests FIRST, as the spec, and let a whole suite - normal, edge, empty - be the thing the agent must satisfy. Now "make the tests pass" means "actually solve the problem".

"Make this failing test pass" - what is the cheapest way a lazy model can satisfy that?

> 📋 **Analogy**
>
> **An exam written before the lecture.** If the questions are fixed and public before anyone teaches the class, you cannot game them by memorizing the teacher's mood - you have to actually know the material. Tests-first pins the target: five assertions (including the annoying edge cases) become a checklist the agent grinds against, regenerating until every box is ticked. A green suite is a real termination signal; a single passing run is not.
> **Where this breaks down:** the exam is only as good as its questions. If the model writes weak tests (misses the empty-list case), the agent will happily pass a weak exam - so test QUALITY is the new failure surface. In practice you review the generated tests, or seed a few of your own, exactly as a good exam board vets its questions.

**📝 `03_tdd_agent.py TDD`** - *Agent*

In [ ]:
# TEST-DRIVEN CODING AGENT - write the tests first, then satisfy them
from google import genai
from google.genai import types

client = genai.Client()
MODEL = "gemini-2.5-flash"

class TDDAgent:
    """Write tests first, then code, iterate until the WHOLE suite passes.
    Runs code in the step-2 sandbox - never on the host."""
    def __init__(self, sandbox, max_retries=3):
        self.sandbox = sandbox               # a DockerSandbox from step 2
        self.max_retries = max_retries

    def _ask(self, prompt):
        return extract_code(client.models.generate_content(
            model=MODEL, contents=prompt).text)

    def _generate_tests(self, task):
        return self._ask(
            "Write Python unit tests using plain assert statements.\n"
            "Cover 5 cases: normal, two edge cases, empty input, and a large input.\n"
            "Assume the function already exists. Return ONLY the test code.\n"
            f"Task: {task}")

    def _generate_code(self, task, tests, error=""):
        extra = f"\nThe tests are failing:\n{error}\nFix the implementation." if error else ""
        return self._ask(
            "Write Python code that makes these tests pass. Return ONLY the function.\n"
            f"Tests:\n{tests}\n\nTask: {task}{extra}")

    def solve(self, task):
        tests = self._generate_tests(task)
        print(f"  Tests generated ({tests.count('assert')} assertions)")
        error = ""
        code = ""
        for attempt in range(self.max_retries):
            code = self._generate_code(task, tests, error)
            full = code + "\n\n" + tests + "\nprint('ALL TESTS PASSED')"
            result = self.sandbox.execute(full)          # sandboxed, not host
            print(f"  Attempt {attempt+1}: {'PASS' if result['success'] else 'FAIL'}")
            if result["success"]:
                return {"code": code, "tests": tests, "attempts": attempt + 1, "passed": True}
            error = result["stderr"]                      # the failing assertions
        return {"code": code, "tests": tests, "attempts": self.max_retries, "passed": False}

agent = TDDAgent(sandbox=DockerSandbox())
r = agent.solve("Write 'flatten' that flattens arbitrarily nested lists, e.g. [1,[2,[3]]] -> [1,2,3]")
print(f"  Passed: {r['passed']} in {r['attempts']} attempt(s)")

# Output:
#   Tests generated (5 assertions)
#   Attempt 1: FAIL     <- passes shallow cases, fails deep nesting [1,[2,[3,[4]]]]
#   Attempt 2: PASS     <- recursion fixed against the failing assertion
#   Passed: True in 2 attempt(s)
#   The suite - not a single run - is the termination signal.


#### 💡 What just happened

⚡ What just happened?The agent wrote the spec as tests FIRST, then ground out an implementation until every assertion passed - deep-nesting bug included. Contrast the cold open: "make the failing test pass" produced `return 180`, but "make this whole SUITE pass" forces a real solution, because you cannot hardcode your way past five cases including the edge ones. **Tests are the termination signal; test QUALITY is the new thing to review.** **When to use TDD vs plain generate-and-fix:** tests-first pays off for anything with real correctness stakes; for a throwaway script the extra test-generation round is cost you may skip - the tradeoff is upfront effort for trustworthy output.

- Scene 1: five assert-based tests are written BEFORE any code - normal, edge, empty. The spec, made executable.

- Scene 2: first implementation - 3 pass green, 2 fail red (deep nesting). Failing tests are a precise to-do list.

- Scene 3: the two red assertions feed back; regenerate; re-run - all five green. A passing suite is the termination signal.

- Scene 4: TDD vs generate-and-hope - a subtly-wrong flatten that RUNS fine but fails a hidden test. Running is not passing.

Controls: Play/Pause, Reset, speed. Scene 4 is the cold open, animated.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Cloud Run Code Executor - Scalable Sandboxed Execution

### Cloud Run Code Executor - Scalable Sandboxed Execution

Deploy code execution as a service. Docker containers on Cloud Run for isolated, scalable agent sandboxes.

One sandbox on your laptop is fine for one agent. Ten agents, or an agent serving many users, needs the sandbox to be a SERVICE - stateless, autoscaling, and physically separate from the agent's brain. Cloud Run runs your step-2 container per request and scales to zero between jobs.

Ten agents each need to run untrusted code at once. What changes vs one agent on your laptop?

> 🏭 **Analogy**
>
> **A bank's cash-counting room versus the teller's drawer.** A teller counting at the window is step 2 - fine for one customer. A high-volume bank sends bulk cash to a separate, hardened counting room with its own doors and cameras (the execution service), so the tellers (agents) never handle the risky bulk directly and the room scales up more counters at rush hour. The agent sends code, the room runs it behind its own walls, and returns only the result.

> ✅ **Info**
>
> ☁ Cloud execution- **Pattern:** Agent sends code → Cloud Run container executes → Returns stdout/stderr/exit code.
> - **Isolation:** Fresh container per execution. No persistent state. Resource limits enforced.
> - **Scaling:** Cloud Run autoscales 0-to-N. Pay only for execution time.

**📝 `04_cloud_executor.py Cloud`** - *Run*

In [ ]:
# 04_cloud_executor.py - the sandbox as a scalable SERVICE (FastAPI + Cloud Run)
# The agent's brain and the code executor live in DIFFERENT places.
from fastapi import FastAPI
from pydantic import BaseModel
import subprocess, tempfile, os

app = FastAPI()

class Job(BaseModel):
    code: str

@app.post("/run")
def run(job: Job):
    # This service IS the sandbox boundary. Deploy it on Cloud Run with a
    # locked-down container; the agent calling it never touches the code.
    with tempfile.TemporaryDirectory() as d:
        path = os.path.join(d, "code.py")
        with open(path, "w") as f:
            f.write(job.code)
        try:
            r = subprocess.run(["python3", path], capture_output=True,
                               text=True, timeout=15)
            return {"success": r.returncode == 0,
                    "stdout": r.stdout[:1000], "stderr": r.stderr[:1000]}
        except subprocess.TimeoutExpired:
            return {"success": False, "stdout": "", "stderr": "timeout"}

# Deploy (each request = a fresh, isolated container instance):
#   gcloud run deploy code-executor --source . --region asia-south1 \
#     --no-allow-unauthenticated --memory 256Mi --timeout 30 --max-instances 20
#
# The agent's _execute() becomes a POST:
#   requests.post(EXECUTOR_URL, json={"code": code}, timeout=30).json()
#
# Output:
#   Service [code-executor] deployed to https://code-executor-xxxx.run.app
#   POST /run {"code": "print(2+2)"}  ->  {"success": true, "stdout": "4\n"}
#   Scales 0-to-N: pay only for execution time, one container per job.


#### 💡 What just happened

What just happened?Cloud-based code execution separates the agent from the sandbox. The agent never runs untrusted code locally - everything executes in ephemeral containers with strict resource limits.

## The Landscape: Who Shipped This Loop, and How

Steps 1-4 are the engine. Steps 5-10 are a field guide to the tools built on that engine - their architectures, sandboxes, benchmarks, costs, and the honest caveats. Treat it as a map that dates quickly, not gospel: the ideas (single-threaded loops, kernel sandboxing, internal evals) outlast the leaderboard numbers.

---

## 🎯 Concept 5: Claude Code Architecture - Single-Threaded Loop, 30+ Tools, Subagents, CLAUDE.md, Hooks, Agent SDK

### Claude Code Architecture - Single-Threaded Loop, 30+ Tools, Subagents, CLAUDE.md, Hooks, Agent SDK

Single-threaded agentic loop. ~20K token overhead. Tools: Read/Write/Edit/Bash/Glob/Grep + Agent (subagents) + LSP + Planning. CLAUDE.md for project memory. 12 hook events for CI/CD. Agent SDK for embedding.

You just built the loop; now see how the most-used terminal coding agent productizes it. The surprising design choice: NOT a swarm of specialized agents, but a single-threaded loop with disciplined tools. Simplicity is the architecture.

Would you expect the best CODING agent to be a swarm of specialists or a single loop?

> 🎭 **Analogy**
>
> **One conductor, not a committee.** An orchestra has dozens of players but ONE conductor holding a single score and beating a single tempo. Claude Code's loop (codenamed "nO") is that conductor: one thread, one message history, gathering context then acting then verifying, over and over. Subagents are like sending the string section to rehearse a hard passage in another room and report back a clean result - they explore on their own context budget and return a compact summary, but they cannot spawn their own subagents (no sub-conductors), which keeps the score readable.
> **Where this breaks down:** some problems genuinely want multiple conductors (independent parallel workstreams) - and that is exactly what multi-agent systems in later lessons explore. The claim here is narrower: for CODING, a single disciplined loop beats a swarm, because coding needs a coherent, auditable thread of changes more than it needs parallelism.

> 📦 **Info**
>
> 🛠 Claude Code internals
> - **Loop:** a single-threaded while-loop (codenamed "nO" - just an internal name): gather context → take action → verify → repeat until the model replies with text and no tool call. One thread, one flat message history ([Anthropic engineering](https://www.anthropic.com/engineering/claude-code)).
> - **Tools:** 30+ tools. Core: Read, Write, Edit (needs prior Read), Bash (persistent session), Glob, Grep, LS. Web: WebFetch, WebSearch. Orchestration: Agent (spawns subagents, cannot nest), EnterPlanMode, TodoWrite, CronCreate. code intelligence via LSP (Language Server Protocol - the same jump-to-definition your IDE uses).
> - **Subagents:** Built-in: Explore (Haiku, read-only), Plan (research), General (full tools). Custom: .claude/agents/ markdown files. Parent → child prompt only; child final message returns verbatim.
> - **CLAUDE.md:** Cascading: ~/.claude/ → PROJECT_ROOT/ → subdirectories. WHY-WHAT-HOW structure. Under 200 lines. Include: build/test/lint commands, coding conventions, key file pointers.
> - **Hooks:** 12 events: PreToolUse, PostToolUse, SessionStart/End, Stop, SubagentStart/Stop, PreCompact, etc. Block writes to .env, auto-format after edits, enforce linting.
> - **Agent SDK:** Python + TypeScript. query() function with allowed_tools, max_turns, permission modes. Embeds Claude Code loop into custom apps.

**📝 `CLAUDE.md Project`** - *Memory*

```
# CLAUDE.md - project memory, loaded every session (keep it under ~200 lines)

## WHAT this repo is
FastAPI service that scores loan applications. Python 3.12, Postgres, pytest.

## HOW to work here
- Build/test:  make test   (runs pytest + ruff; MUST pass before any PR)
- Never write to .env or migrations/ without asking.
- Money math uses Decimal, never float. New endpoints need a test.

## WHERE things live
- Scoring logic:  app/scoring/     Routes: app/api/     Fixtures: tests/conftest.py

# The agent reads this at startup, so it stops re-discovering your conventions
# every session - the single highest-ROI file in an agent-run repo.
#
# Output: (what the agent does with it)
#   session start -> reads CLAUDE.md -> uses `make test`, Decimal money, app/ layout
#   without re-grepping the repo to rediscover any of it.
```

#### 💡 What just happened

What just happened?Claude Code proves that **simplicity scales**: a single disciplined loop beats a multi-agent swarm for coding, because coding wants one coherent, auditable thread of edits. CLAUDE.md (the project memory file loaded each session) is the highest-ROI habit - it stops the agent re-exploring your repo from scratch every time. **When a swarm IS worth it:** genuinely independent parallel workstreams - which later Module 11 lessons cover.

---

## 🎯 Concept 6: Cursor - IDE-as-Agent: Tab, Agent Mode, Composer, Rules, Cloud Agents

### Cursor - IDE-as-Agent: Tab, Agent Mode, Composer, Rules, Cloud Agents

VS Code fork. Tab: next-action prediction (not just autocomplete). Agent Mode: multi-file + terminal + browser. Composer: persistent context. Cloud Agents: async VMs. .cursor/rules/ for project context.

Claude Code lives in the terminal; Cursor puts the same loop inside the editor. The interesting shift is that the IDE stops being where you type and becomes where you SUPERVISE - Tab predicts your next edit, Agent Mode makes multi-file changes, and Cloud Agents go do work while you are away.

What becomes the developer's core skill once the IDE can make multi-file edits for you?

> 🧰 **Analogy**
>
> **Power steering, then self-parking.** Autocomplete was cruise control - it held the line you were already driving. Cursor's Tab is power steering that anticipates the turn (it predicts your NEXT edit, jumps the cursor, edits across files). Agent Mode is the car parking itself while you watch. The driver's job moves from steering to deciding where to go and confirming the car did it right - which is why "review the diff" becomes the core skill, not "type the code". **When the IDE agent fits vs a terminal agent:** IDE tools win for interactive, in-the-loop editing; deep autonomous runs are often better in a terminal or cloud agent - the tradeoff is supervision bandwidth versus hands-off delegation.

> ✅ **Info**
>
> 🖥 Cursor architecture
> - **Tab:** Custom Fusion model. Multi-line prediction, diff popups, cursor jumps, cross-file edits. "Tab-tab-tab" flow.
> - **Agent Mode:** Reads codebases, plans changes, executes multi-file, runs terminal, native browser, MCP integration. YOLO Mode: auto-execute without confirmation.
> - **Composer 2:** Persistent context across sessions. 3-phase: explore → plan → execute. Custom Composer model optimized for diffs.
> - **Rules:** .cursor/rules/ with MDC format + YAML frontmatter. alwaysApply, globs patterns, @file references. Equivalent to CLAUDE.md for Cursor.
> - **Cloud Agents:** Isolated VMs. Clone, build, test, PR - offline. a large and growing share of internal PRs from cloud agents; self-hosted options for enterprise.
> - **Pricing & scale:** Pro around $20/mo (unlimited Tab); Teams tiers above that. Anysphere (Cursor's maker) grew explosively - roughly $29B valuation in late 2025, and as of mid-2026 reported to be acquired by SpaceX at about $60B. The dollar figures date fast; the product pattern does not (source: [TechCrunch](https://techcrunch.com/2026/04/17/sources-cursor-in-talks-to-raise-2b-at-50b-valuation-as-enterprise-growth-surges/)).

**📝 `.cursor/rules/python.mdc Cursor`** - *Rules*

```
# .cursor/rules/python.mdc - Cursor's project rules (MDC = markdown + YAML frontmatter)
---
description: Python conventions for this repo
alwaysApply: true
globs: ["**/*.py"]
---
- Use type hints on every public function. Format with ruff.
- Money is Decimal, never float. New endpoints need a pytest test.
- Prefer standard library; ask before adding a dependency.

# Same job as CLAUDE.md and AGENTS.md, in Cursor's format. Every serious
# agent tool has one of these - it is how you stop re-explaining your repo.
#
# Output: (Cursor applies it to every *.py edit)
#   agent edit -> type hints added, ruff formatting, Decimal for money,
#   no new dependency introduced without asking first.
```

#### 💡 What just happened

What just happened?Recent Cursor versions push the IDE into the background and the agent to the front. Cloud Agents enable async delegation (clone, build, test, PR while you are away). The durable insight: **for most developers, fast in-editor prediction plus an agent mode covers the large majority of daily coding** - the exact split varies by team.

---

## 🎯 Concept 7: Codex CLI + Devin - Open-Source Sandboxed CLI, Cloud-Native Autonomous Agent, AGENTS.md Standard

### Codex CLI + Devin - Open-Source Sandboxed CLI, Cloud-Native Autonomous Agent, AGENTS.md Standard

Codex: Rust, kernel sandboxing (Bubblewrap+seccomp), open-source, AGENTS.md. Devin: autonomous cloud agent, ACU-billed, folded Windsurf into Devin Desktop.

Two contrasting bets on the same loop. Codex CLI is the open, security-first terminal agent - it puts the sandbox lesson from step 2 into the KERNEL. Devin is the fully-autonomous cloud teammate you delegate a whole ticket to. One trades autonomy for auditability; the other bets on hands-off delegation.

Codex enforces its sandbox in the kernel; Devin runs autonomously in the cloud. Which needs re-verifying every task?

> 🔑 **Analogy**
>
> **A locksmith's workshop vs a valet service.** Codex is the locksmith who works at your bench with the tools bolted down and the doors monitored at the frame (kernel-level Bubblewrap/seccomp/Landlock - the sandbox enforced by the building, not by a polite sign). Devin is the valet: you hand over the keys and the task ("migrate this service"), and it drives off, does the work in its own garage (a cloud Devbox VM), and returns a finished PR. The valet is magical when the task is well-scoped and a liability when the instructions are vague - "senior at understanding, junior at execution".
> **Where this breaks down:** a locksmith and a valet are people you can trust once vetted; an autonomous agent must be RE-verified every task, because it will confidently drive the wrong car if the ticket is ambiguous. Delegation scales the work, not the trust - which is why merge rates and human review gates matter more than raw autonomy.

> 📦 **Info**
>
> 🚀 Codex + Devin
> - **Codex CLI:** open-source, written in Rust, ~67K GitHub stars. Kernel-level sandboxing - Bubblewrap + seccomp + Landlock on Linux, Apple Seatbelt on macOS, plus a Windows sandbox (Linux kernel features that filter what code can touch, tighter than a container). Read-only filesystem and network off by default. GPT-5.4 default (~272K context). AGENTS.md config standard. Terminal-Bench scores lead the pack, though exact numbers move release to release (source: [OpenAI Codex CLI docs](https://developers.openai.com/codex/cli)).
> - **AGENTS.md:** a cross-tool convention (a markdown file of project instructions, like CLAUDE.md but vendor-neutral) read by Codex, Cursor, Amp, Jules and others. Loaded hierarchically; capped around 32 KiB per file in Codex (extra content is silently truncated).
> - **Devin:** an autonomous cloud teammate (a planner plus its own Devbox VM). Strongest on well-scoped, repetitive work (migrations, security fixes, test generation); billed in Agent Compute Units (ACUs - Devin's usage unit) on free and paid tiers (Pro around $20, higher tiers above). Cognition (Devin's maker) folded the acquired Windsurf editor into "Devin Desktop".
> - **Devin limitations:** needs clear requirements upfront and handles mid-task scope changes poorly - "senior at understanding, junior at execution". Early independent tests found low success rates on open-ended tasks; it shines on narrow, well-specified ones.

**📝 `AGENTS.md Agent`** - *Config*

```
# AGENTS.md - vendor-neutral agent instructions (Codex, Cursor, Amp, Jules, ...)
# Same idea as CLAUDE.md, read by many tools. Loaded hierarchically; ~32 KiB cap in Codex.

## Setup
Run `uv sync` before anything. Tests: `pytest -q`. Lint: `ruff check .`.

## Conventions
- Small, reviewable commits. Explain WHY in the body.
- Do not touch infra/ or secrets. Ask before adding a dependency.

# One file, every agent that speaks the standard - so you write your project's
# rules once instead of once per tool.
#
# Output: (Codex/Cursor/Jules all read the same file)
#   agent start -> runs `uv sync`, then `pytest -q` / `ruff check .` as told,
#   and will not touch infra/ or add a dependency without asking.
```

#### 💡 What just happened

What just happened?Codex shows open-source + kernel sandboxing can match proprietary tools on autonomy AND safety. Devin shows fully-autonomous agents work for **specific, well-scoped tasks** but struggle with open-ended development - **delegation scales the work, not the trust**, so merge rates and human review gates matter more than raw autonomy. AGENTS.md is converging into a shared agent-config standard.

---

## 🎯 Concept 8: Ecosystem - Copilot (20M Users), Cline (BYOM), Aider (Git-Native), OpenHands, Augment Code, Jules

### Ecosystem - Copilot (20M Users), Cline (BYOM), Aider (Git-Native), OpenHands, Augment Code, Jules

Copilot: Coding Agent for async PRs. Cline: open-source, human-in-the-loop, any LLM. Aider: terminal + git-native. OpenHands: Docker sandboxed. Augment: enterprise code indexing. Jules: Google Cloud VMs.

Beyond the headline three, a dozen tools stake out corners of the same map. You do not need to memorize them - you need the AXES they vary on: open vs closed, terminal vs IDE vs cloud, human-approves-every-edit vs full autonomy, bring-your-own-model vs locked. Place any new tool on those axes and you already understand it.

A new coding tool launches. What is the fastest way to understand it?

> 🧭 **Analogy**
>
> **A menu, not a monolith.** Coding tools are a restaurant menu organized by a few tastes: how much autonomy (appetizer of autocomplete vs full-course autonomous agent), how open (house recipe vs bring-your-own-model), where it runs (dine-in IDE, takeaway terminal, delivery cloud). Cline is "you approve every bite" (human-in-the-loop, any model, even a local one); Aider is "git-native, auto-commits each course"; Copilot is the popular chain everyone has tried. Same nourishment, different service style - pick by appetite and budget, not by hype.

> ✅ **Info**
>
> 🌐 Ecosystem
> - **GitHub Copilot:** the most widely deployed (tens of millions of users, most of the Fortune 100). Its Coding Agent takes an issue and returns a draft PR (branch → code → test → PR). From about $10/mo.
> - **Cline:** millions of developers. Open-source VS Code extension. HITL: every edit needs approval. Works with ANY LLM (including local Ollama). MCP in core loop.
> - **Aider:** a large open-source following. Terminal + git-native (auto-commits). Architect/Editor mode (plan model + implement model). Polyglot benchmark across several languages.
> - **OpenHands:** a popular MIT-licensed project. Docker sandboxed. Event-stream architecture. OpenHands Index: composite benchmark (5 categories).
> - **Augment Code:** a context engine that indexes very large codebases (hundreds of thousands of files). reports markedly fewer hallucinations via deep code indexing; among the SWE-bench Pro leaders; ISO/IEC 42001 certified.
> - **Google Jules:** Async: clone repo → plan → PR. a free daily task quota.

#### 💡 What just happened

What just happened?The ecosystem sorts onto three axes worth memorizing instead of the tool names: **terminal-native** (Claude Code, Codex, Aider) for deep autonomy, **IDE-integrated** (Cursor, Copilot, Cline) for interactive velocity, **cloud-native** (Devin, Jules) for async delegation - crossed with open-vs-closed and approve-every-edit-vs-full-autonomy. Place any new tool on those axes and you already understand it.

---

## 🎯 Concept 9: SWE-bench + Benchmarks - Verified (Contaminated), Pro, LiveCodeBench, Terminal-Bench, Aider Polyglot, Internal Evals

### SWE-bench + Benchmarks - Verified (Contaminated), Pro, LiveCodeBench, Terminal-Bench, Aider Polyglot, Internal Evals

SWE-bench Verified: once the top number, now contaminated and abandoned. Pro: multi-language. LiveCodeBench: contamination-resistant. Build internal evals from YOUR codebase.

This is the most important step for making real decisions - and the most misused. Public leaderboards are contaminated: models were trained on the very repos the benchmark tests, so a high score can mean "memorized the answer", not "can code". The skill is reading benchmarks like a skeptic and building your own.

An agent posts a very high score on a public coding benchmark. Your first reaction?

> 🏆 **Analogy**
>
> **An open-book exam where the students wrote the answer key.** If last year's exam (and its solutions) leaked into everyone's study notes, a near-perfect average measures memory, not understanding - and that is exactly what happened to SWE-bench Verified (the fix was often hinted in the issue's own comments, and the repos were in the training data). The honest test is a POP quiz on fresh material: contamination-resistant benchmarks rotate in new problems, and the scores drop sharply - the same agents that "ace" the leaked exam struggle on the pop quiz.
> **Where this breaks down:** even a pop quiz is not YOUR job. The only exam that predicts performance on your codebase is one written from your codebase - 20 known bugs and features from your own repo. A dull-sounding internal eval beats every glossy leaderboard, because it cannot be memorized in advance.

> 📦 **Info**
>
> 📊 Benchmarks
> - **SWE-bench Verified:** 500 Python tasks, mostly bug fixes. Once the headline number (frontier models ~70-80%), now discredited: OpenAI formally stopped using it on 2026-02-23 after finding a large share of failed tests were flawed AND every frontier model showed training-data contamination (the repos, and often the fix hinted in issue comments, were in training). Honestly measured on contamination-resistant benchmarks, the same models score far lower (source: [OpenAI](https://openai.com/index/why-we-no-longer-evaluate-swe-bench-verified/)).
> - **SWE-bench Pro:** a larger, multi-language, contamination-resistant successor; top scores sit far below the old Verified numbers (roughly the 50s%), which is the honest signal - the drop is contamination lifting, not models getting worse.
> - **LiveCodeBench / SWE-bench Live:** continuously rotate in problems published AFTER training cutoffs, so memorization does not pay. The most trustworthy public signal, though still not YOUR codebase.
> - **Terminal-Bench:** real command-line tasks (run a build, fix an env), a better proxy for agent work than isolated bug fixes. Leaders trade places release to release.
> - **Aider Polyglot:** 225 exercises across 6 languages, testing raw multi-language editing ability rather than repo navigation.
> - **CRITICAL:** Use one floor benchmark + one production benchmark + internal evals from YOUR codebase for real decisions.

#### 💡 What just happened

What just happened?No single benchmark captures real coding ability. SWE-bench Verified is increasingly unreliable. The practical approach: **build internal evals from your own codebase**. A simple "can the agent fix these 20 known bugs in our repo" test tells you more than any public leaderboard. **The tradeoff:** public benchmarks are free and comparable but contaminated and generic; an internal eval costs you a day to build and predicts YOUR results - use a contamination-resistant public benchmark as a floor and your own eval for the real decision.

- Scene 1: the headline - an agent at ~80% on SWE-bench Verified, a shiny trophy.

- Scene 2: the leak - the fix was hinted in the issue comments and the repo was in training data; a "memorized" stamp lands.

- Scene 3: the gap - Verified ~80% next to a contamination-resistant benchmark far lower; the same agent, honestly measured.

- Scene 4: build your own - 20 tasks from YOUR repo (bugs, features, refactors); two agents get honest, comparable scores.

Controls: Play/Pause, Reset, speed. Scene 3 is why OpenAI abandoned Verified in Feb 2026.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 10: Production Patterns & India - Edit-Test-Debug Loop, Git Workflows, CI/CD, Sandboxing, Cost, Indian Adoption

### Production Patterns & India - Edit-Test-Debug Loop, Git Workflows, CI/CD, Sandboxing, Cost, Indian Adoption

Tests as termination signals. Git worktrees for parallel agents. GitHub Agentic Workflows. Kernel-level sandboxing. Model routing for large cost savings. India: high adoption, hybrid multi-vendor stacks.

Everything so far, assembled into how teams actually run coding agents in production - and what that looks like specifically in the Indian market most of this course's builders ship to. Three patterns are non-negotiable: tests as the stop signal, real sandboxing, and cost guardrails.

Your CI lets an agent fix failing tests and open PRs. What is the one guardrail you must NOT skip?

> 🏭 **Analogy**
>
> **A factory floor, not a workshop.** A hobbyist runs one tool by hand; a factory runs many in parallel with safety interlocks. Git worktrees let several agents work isolated lanes at once (parallel benches). CI/CD splits the risky part: the agent PRODUCES a patch in a locked room, and a separate job with scoped permissions APPLIES it (the machine that makes the part is not the machine that installs it). Kernel sandboxes are the interlocks that stop a runaway tool. And cost routing - cheap models for most of the grunt work, frontier models only for the hard reasoning - is the factory keeping its energy bill sane.
> **Where this breaks down:** a factory optimizes a known process; coding agents are still new enough that the "process" shifts monthly. Indian teams lead here precisely by staying pragmatic - a reported majority run HYBRID stacks (multiple vendors, open-source tools for budget lanes) rather than betting the floor on one machine. The patterns are stable; the specific tools on the floor are not.

> ✅ **Info**
>
> 🇮🇳 Production + India
> - **Edit-test-debug:** Plan → Execute → Verify. Tests are termination signals. Per-file test commands for fast feedback. Agentic search (grep+glob) over RAG for codebase discovery.
> - **Git:** Worktrees for parallel agents (/batch = 10x speedup). Co-authored commits for traceability. PR-based code review (Claude Code Review, CodeRabbit).
> - **CI/CD:** GitHub Agentic Workflows: agent produces artifact → separate job applies with scoped permissions. Claude Code GitHub Actions via @claude. Codex CLI with --never approval.
> - **Sandboxing:** Kernel-level (Bubblewrap/Seatbelt) > MicroVMs (Firecracker/Kata) > Docker (insufficient alone). Real incidents: rm -rf ~/, /proc escape, npm token exfiltration.
> - **Cost:** heavy Claude Code use runs on the order of low tens of dollars per developer per active day. Model routing (cheap models for grunt work) plus prompt caching (big discounts on cached tokens) together cut that substantially - reported to be substantial (often a majority of spend).
> - **India:** adoption among large Indian tech firms is very high (industry surveys report the large majority live in production), and the big services companies hold hundreds of thousands of Copilot licenses combined. Budget-conscious teams lean on the open-source stack (Cline + Aider + OpenHands) with bring-your-own-model.

**📝 `routing.py Cost`** - *Routing*

In [ ]:
# routing.py - send cheap work to a cheap model, hard reasoning to a frontier one
def pick_model(action: str) -> str:
    CHEAP = "gemini-2.5-flash"          # file reads, greps, small edits (~60-70% of actions)
    FRONTIER = "gemini-2.5-pro"         # architecture, tricky debugging, multi-file refactors
    cheap_actions = {"read", "grep", "glob", "format", "rename"}
    return CHEAP if action in cheap_actions else FRONTIER

# Parallel agents without stepping on each other - one git worktree per agent:
#   git worktree add ../wt-fix-auth  -b fix-auth
#   git worktree add ../wt-add-tests -b add-tests
# Each agent works an isolated checkout; merge the branches when green.
#
# Output:
#   pick_model("grep")   -> gemini-2.5-flash    (the cheap lane, most actions)
#   pick_model("refactor") -> gemini-2.5-pro    (frontier only when it earns it)
#   Routing cheap work away from the frontier model is where the 60-80% savings come from.

#### 💡 What just happened

What just happened?Three non-negotiable production patterns: **(1) tests as termination signals**, (2) real sandboxing (kernel-level or microVM for higher-risk work, not a bare subprocess), (3) model routing with cost guardrails. Indian teams are among the fastest adopters, favoring pragmatic hybrid multi-vendor stacks over betting on a single tool.

## Recap

> 📦 **Info**
>
> What we built and mapped
> - **The loop** - generate, execute, read the traceback, fix, retry; the error is the observation (Reflexion applied to code).
> - **The sandbox ladder** - bare subprocess (unsafe) -> Docker (right default) -> kernel (Bubblewrap/seccomp) -> microVM (Firecracker); never run model code on your host.
> - **Tests-first** - a passing suite is the real termination signal; a single green run is not (the flat-180 trap).
> - **The landscape** - single-threaded loops (Claude Code), IDE-as-agent (Cursor), kernel-sandboxed CLI (Codex), autonomous cloud (Devin), and a menu of the rest - placed on the open/terminal/autonomy axes.
> - **Honest evaluation** - why SWE-bench Verified was abandoned, and why a 20-task eval from your own repo is the only score that predicts your results.

> ✅ **Info**
>
> 🔗 Where this goes next
> - **Lesson 11.6 (Computer Use Agents)** generalized this loop from a code executor to a whole screen - we build on that sandbox-everything instinct here.
> - **In Lesson 11.8 (Deep Research Agents)**, the same generate-verify loop points at the web and documents instead of a test suite.
> - **In Module 14 (LLMOps)**, the internal-eval idea from step 9 grows into a full eval-driven development discipline.

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Coding Agents- Generate, Run, Fix**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-11_7.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-11.7.html` - regenerate this notebook whenever the source HTML is updated.*
